# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

## 1. My rule and its reason codes

The baseline rule: **rank pages by raw search impressions**, filtered to pages sitting in positions 4–15 with at least 1,000 impressions in March 2026.

The logic is that pages in that position range are getting real visibility but aren't ranking well enough to convert it into clicks — they're the ones where a content refresh has the most potential upside. Pages already in top 3 probably don't need help, and pages past position 15 might need more than just a content update.

**Reason codes:**
- `HIGH_VISIBILITY` — meets both thresholds; flagged for review
- `LOW_PRIORITY` — below thresholds; monitor but no urgent action

This is deliberately simple. The point is to have a concrete, explainable number to compare against whatever the model produces later.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Build a baseline action score

baseline = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    CASE
        WHEN gsc_impressions >= 1000 AND gsc_avg_position BETWEEN 4 AND 15
            THEN gsc_impressions
        ELSE 0
    END AS score,

    CASE
        WHEN gsc_impressions >= 1000 AND gsc_avg_position BETWEEN 4 AND 15
            THEN 'HIGH_VISIBILITY'
        ELSE 'LOW_PRIORITY'
    END AS reason_code,

    CASE
        WHEN gsc_impressions >= 1000 AND gsc_avg_position BETWEEN 4 AND 15
            THEN 'Review for Content Refresh'
        ELSE 'Monitor'
    END AS action

FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

baseline = baseline.sort_values("score", ascending=False)

baseline.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action
103661,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,37368,HIGH_VISIBILITY,Review for Content Refresh
8860963,2026-03-29,client_23a62021009f63c4,content_66288edeb93b7c4f,24577,66,10.794239,24577,HIGH_VISIBILITY,Review for Content Refresh
883810,2026-03-05,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,24456,1,4.066814,24456,HIGH_VISIBILITY,Review for Content Refresh
9650754,2026-03-28,client_23a62021009f63c4,content_66288edeb93b7c4f,23542,165,11.112140,23542,HIGH_VISIBILITY,Review for Content Refresh
4337450,2026-03-15,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,16454,1,5.131579,16454,HIGH_VISIBILITY,Review for Content Refresh


In [5]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

CSV written successfully.


## 3. Top-20 review

Looking at the top 20 in the ranked queue:

- Most are pages with impression counts in the 10k–37k range during March, sitting around positions 5–15.
- All 20 got the same reason code (`HIGH_VISIBILITY`) and the same action (`Review for Content Refresh`) — because the rule has no nuance, it treats them identically.
- A couple of content hashes appear multiple times on different dates (e.g. `content_66288edeb93b7c4f` shows up twice). That's a ranking artifact from the per-day grain — the same page on different days gets scored separately. In a real queue you'd deduplicate by page.

**Where this baseline could be wrong:**
- It has no idea whether a page was already updated last week — it would still flag it
- High impressions with zero clicks might mean the page is appearing for irrelevant queries, not that a content refresh would help
- Position 8 on a niche query is very different from position 8 on a high-volume one, but the rule treats them the same

These are the failure modes the model should improve on.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = baseline.head(20)

top20[[
    "content_hash_id",
    "score",
    "reason_code",
    "action",
    "gsc_impressions",
    "gsc_avg_position"
]]


,content_hash_id,score,reason_code,action,gsc_impressions,gsc_avg_position
103661,content_945d6ff91386c817,37368,HIGH_VISIBILITY,Review for Content Refresh,37368,8.613948
8860963,content_66288edeb93b7c4f,24577,HIGH_VISIBILITY,Review for Content Refresh,24577,10.794239
883810,content_1642f339bd6e7c8d,24456,HIGH_VISIBILITY,Review for Content Refresh,24456,4.066814
9650754,content_66288edeb93b7c4f,23542,HIGH_VISIBILITY,Review for Content Refresh,23542,11.112140
4337450,content_1642f339bd6e7c8d,16454,HIGH_VISIBILITY,Review for Content Refresh,16454,5.131579
9653508,content_e943d753806d7af3,15522,HIGH_VISIBILITY,Review for Content Refresh,15522,8.788429
9343778,content_046fc480045b88f5,14185,HIGH_VISIBILITY,Review for Content Refresh,14185,7.050335
103514,content_0c5606abaaab3178,13827,HIGH_VISIBILITY,Review for Content Refresh,13827,4.112533
8095571,content_046fc480045b88f5,13764,HIGH_VISIBILITY,Review for Content Refresh,13764,6.915650
268027,content_7c6373141eae744a,12648,HIGH_VISIBILITY,Review for Content Refresh,12648,6.159235


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some highly ranked pages may not actually require content updates because the rule only considers search impressions and average position. Pages with seasonal traffic or temporary ranking changes may appear more important than they really are.

No future information or label-derived fields were used when building this baseline. The rule relies only on observable search performance metrics that are available at the decision time. Therefore, no data leakage was intentionally introduced into this baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.